# Kaggriculture FarmOps Guardian Agent

Capstone project for the 5-Day AI Agents: Intensive Vibe Coding Course With Google.

This notebook demonstrates an autonomous farming resource-management agent with tools, memory, guardrails, evaluation, and trace logging.

## Requirement mapping

| Course concept | Evidence |
|---|---|
| Agent / multi-agent system using ADK | `adk_agent.py` in the repository |
| MCP Server | `mcp_server.py` in the repository |
| Security features | Deterministic guardrails in this notebook and `guardrails.py` |
| Agent tools / skills | Risk, cost, irrigation, and yield tools |
| Memory and context | `AgentMemory` and recent action context |
| Evaluation | Five scenario tests with expected actions |
| Deployability | FastAPI app and Dockerfile in the repository |

## Agent architecture

Farm State → Tool Layer → Planner Agent → Guardrail Validator → Final Action → Memory Store → Evaluation and Trace Log

In [ ]:

from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Any
from datetime import datetime, timezone
import pandas as pd

ALLOWED_ACTIONS = {"IRRIGATE", "INSPECT", "FERTILIZE", "HARVEST", "WAIT"}

@dataclass
class FarmState:
    day: int
    crop: str
    growth_stage: str
    soil_moisture_pct: float
    temperature_c: float
    rain_forecast_mm: float
    disease_signals: int
    water_available_liters: float
    budget_usd: float
    crop_maturity_pct: float
    last_action: str = "NONE"

def estimate_irrigation_need(state: FarmState) -> Dict[str, Any]:
    target = 65
    deficit = max(0, target - state.soil_moisture_pct)
    liters_needed = round(deficit * 12, 1)
    urgency = "high" if state.soil_moisture_pct < 35 and state.rain_forecast_mm < 5 else "medium" if state.soil_moisture_pct < 55 else "low"
    return {"tool": "estimate_irrigation_need", "liters_needed": liters_needed, "urgency": urgency}

def estimate_weather_risk(state: FarmState) -> Dict[str, Any]:
    heat_risk = state.temperature_c >= 32
    drought_risk = state.rain_forecast_mm < 3 and state.soil_moisture_pct < 45
    risk_score = int(heat_risk) + int(drought_risk)
    label = "high" if risk_score == 2 else "medium" if risk_score == 1 else "low"
    return {"tool": "estimate_weather_risk", "risk": label, "heat_risk": heat_risk, "drought_risk": drought_risk}

def estimate_disease_risk(state: FarmState) -> Dict[str, Any]:
    score = min(100, state.disease_signals * 25 + (10 if state.rain_forecast_mm > 15 else 0))
    label = "high" if score >= 60 else "medium" if score >= 30 else "low"
    return {"tool": "estimate_disease_risk", "score": score, "risk": label}

def estimate_action_cost(action: str, state: FarmState) -> Dict[str, Any]:
    base_costs = {"IRRIGATE": 15, "INSPECT": 10, "FERTILIZE": 40, "HARVEST": 75, "WAIT": 0}
    cost = base_costs[action]
    return {"tool": "estimate_action_cost", "action": action, "cost_usd": cost, "within_budget": cost <= state.budget_usd}

def estimate_yield_impact(action: str, state: FarmState) -> Dict[str, Any]:
    impact = 0
    if action == "IRRIGATE" and state.soil_moisture_pct < 55:
        impact += 18
    if action == "INSPECT" and state.disease_signals >= 2:
        impact += 12
    if action == "FERTILIZE" and state.growth_stage in ["vegetative", "flowering"]:
        impact += 15
    if action == "HARVEST" and state.crop_maturity_pct >= 90:
        impact += 25
    if action == "WAIT":
        impact += 2 if state.soil_moisture_pct >= 55 and state.disease_signals == 0 else -8
    return {"tool": "estimate_yield_impact", "action": action, "yield_impact_score": impact}

class AgentMemory:
    def __init__(self):
        self.history: List[Dict[str, Any]] = []

    def remember(self, record: Dict[str, Any]) -> None:
        self.history.append(record)

    def recent_actions(self, n: int = 3) -> List[str]:
        return [item["final_action"] for item in self.history[-n:]]

    def as_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(self.history)

def validate_action(action: str, state: FarmState, tool_results: Dict[str, Any], recent_actions: List[str]) -> Tuple[str, List[str]]:
    reasons = []
    final_action = action

    if action not in ALLOWED_ACTIONS:
        return "WAIT", [f"Blocked unknown action: {action}."]

    cost = estimate_action_cost(action, state)["cost_usd"]
    irrigation = tool_results["irrigation"]

    if cost > state.budget_usd:
        reasons.append(f"Blocked {action}: cost ${cost} exceeds budget ${state.budget_usd}.")
        final_action = "WAIT"

    if action == "IRRIGATE" and irrigation["liters_needed"] > state.water_available_liters:
        reasons.append("Blocked IRRIGATE: not enough water available.")
        final_action = "INSPECT"

    if action == "HARVEST" and state.crop_maturity_pct < 90:
        reasons.append("Blocked HARVEST: crop maturity is below 90%.")
        final_action = "WAIT"

    if action == "FERTILIZE" and state.growth_stage not in ["vegetative", "flowering"]:
        reasons.append("Blocked FERTILIZE: crop is not in the correct growth stage.")
        final_action = "WAIT"

    if recent_actions.count("IRRIGATE") >= 2 and action == "IRRIGATE":
        reasons.append("Blocked IRRIGATE: too many recent irrigation actions.")
        final_action = "INSPECT"

    if not reasons:
        reasons.append("Action passed guardrails.")

    return final_action, reasons

class FarmOpsGuardianAgent:
    def __init__(self, memory: AgentMemory | None = None):
        self.memory = memory or AgentMemory()

    def observe(self, state: FarmState) -> Dict[str, Any]:
        return {"state": asdict(state), "recent_actions": self.memory.recent_actions()}

    def call_tools(self, state: FarmState) -> Dict[str, Any]:
        return {
            "irrigation": estimate_irrigation_need(state),
            "weather": estimate_weather_risk(state),
            "disease": estimate_disease_risk(state),
        }

    def plan(self, state: FarmState, tools: Dict[str, Any]) -> Tuple[str, str]:
        disease = tools["disease"]["risk"]
        weather = tools["weather"]["risk"]
        irrigation = tools["irrigation"]

        if state.crop_maturity_pct >= 90 and state.budget_usd >= 75:
            return "HARVEST", "Crop maturity is high and budget allows harvest."
        if irrigation["urgency"] == "high":
            return "IRRIGATE", "Soil moisture is low and rain forecast is limited."
        if disease == "high":
            return "INSPECT", "Disease signal is high, so inspection is needed before treatment."
        if state.growth_stage in ["vegetative", "flowering"] and state.budget_usd >= 40 and disease != "high":
            return "FERTILIZE", "Growth stage is appropriate and budget allows fertilization."
        if weather == "high":
            return "IRRIGATE", "Weather risk suggests preventing crop stress."
        return "WAIT", "Conditions are stable, so waiting avoids unnecessary cost."

    def run(self, state: FarmState) -> Dict[str, Any]:
        observation = self.observe(state)
        tools = self.call_tools(state)
        proposed_action, rationale = self.plan(state, tools)
        final_action, guardrail_reasons = validate_action(
            proposed_action, state, tools, self.memory.recent_actions()
        )

        trace = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "day": state.day,
            "crop": state.crop,
            "growth_stage": state.growth_stage,
            "soil_moisture_pct": state.soil_moisture_pct,
            "rain_forecast_mm": state.rain_forecast_mm,
            "disease_signals": state.disease_signals,
            "budget_usd": state.budget_usd,
            "proposed_action": proposed_action,
            "final_action": final_action,
            "rationale": rationale,
            "guardrails": " | ".join(guardrail_reasons),
            "tool_results": tools,
            "observation": observation,
        }
        self.memory.remember(trace)
        return trace


## Demo scenarios

The scenarios are designed to demonstrate irrigation, inspection, harvest, guardrail override, and budget-aware waiting.

In [ ]:

scenarios = [
    FarmState(day=1, crop="tomato", growth_stage="vegetative", soil_moisture_pct=28, temperature_c=35, rain_forecast_mm=0, disease_signals=0, water_available_liters=800, budget_usd=120, crop_maturity_pct=20),
    FarmState(day=2, crop="tomato", growth_stage="flowering", soil_moisture_pct=67, temperature_c=27, rain_forecast_mm=8, disease_signals=3, water_available_liters=600, budget_usd=80, crop_maturity_pct=55),
    FarmState(day=3, crop="tomato", growth_stage="mature", soil_moisture_pct=58, temperature_c=24, rain_forecast_mm=5, disease_signals=0, water_available_liters=500, budget_usd=100, crop_maturity_pct=94),
    FarmState(day=4, crop="tomato", growth_stage="seedling", soil_moisture_pct=30, temperature_c=33, rain_forecast_mm=0, disease_signals=0, water_available_liters=10, budget_usd=90, crop_maturity_pct=5),
    FarmState(day=5, crop="tomato", growth_stage="vegetative", soil_moisture_pct=70, temperature_c=26, rain_forecast_mm=10, disease_signals=0, water_available_liters=700, budget_usd=10, crop_maturity_pct=30),
]

agent = FarmOpsGuardianAgent()
results = [agent.run(s) for s in scenarios]
df = pd.DataFrame(results)
df[["day", "crop", "growth_stage", "proposed_action", "final_action", "rationale", "guardrails"]]


## Evaluation

The evaluation checks whether the final action matches the expected action for each scenario.

In [ ]:

expected = {
    1: "IRRIGATE",
    2: "INSPECT",
    3: "HARVEST",
    4: "INSPECT",
    5: "WAIT",
}

eval_rows = []
for row in results:
    day = row["day"]
    eval_rows.append({
        "day": day,
        "expected_action": expected[day],
        "final_action": row["final_action"],
        "passed": row["final_action"] == expected[day],
        "guardrails": row["guardrails"]
    })

eval_df = pd.DataFrame(eval_rows)
print(f"Scenario accuracy: {eval_df['passed'].mean():.0%}")
eval_df


## Observability trace

The trace shows proposed actions, final actions, and guardrail results.

In [ ]:

trace_df = agent.memory.as_dataframe()
trace_df[["timestamp", "day", "proposed_action", "final_action", "guardrails"]]


## ADK-style multi-agent design

The repository includes `farmops_guardian/adk_agent.py`, which defines:

- `farmops_guardian`, the coordinator agent.
- `agronomy_agent`, focused on crop health and yield impact.
- `resource_agent`, focused on water and budget.

The notebook runs without external keys, but the repository contains the ADK implementation sketch for the capstone requirement.


## MCP server

The repository includes `farmops_guardian/mcp_server.py`, a local MCP server that exposes the farm tools as MCP tools:

- `irrigation_need`
- `weather_risk`
- `disease_risk`

This shows how the same agent skills can be made interoperable through MCP.


## Deployability

The repository includes:

- `app.py`, a FastAPI wrapper around the agent.
- `Dockerfile`, a container build file.
- `DEPLOYMENT_NOTES.md`, local and container run instructions.

This demonstrates how the notebook prototype can become a service.


## Conclusion

FarmOps Guardian demonstrates an end-to-end agentic pattern: observe, call tools, plan, validate, remember, and evaluate. It is intentionally runnable as a Kaggle notebook while also including production-oriented code for ADK, MCP, guardrails, API deployment, and Docker deployment.